In [1]:
import sqlite3
import pandas as pd
# Kết nối SQLite
conn = sqlite3.connect("students.db")
cursor = conn.cursor()


In [2]:
cursor.execute("DROP TABLE IF EXISTS student;")
cursor.execute("DROP TABLE IF EXISTS course;")

# Tạo bảng student
cursor.execute("""
CREATE TABLE student (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
);
""")

# Tạo bảng course
cursor.execute("""
CREATE TABLE course (
    id INTEGER PRIMARY KEY,
    course_name TEXT
);
""")

# Lưu thay đổi vào database
conn.commit()

In [3]:
# Dữ liệu cho bảng student
student_data = [
    (1, "Nguyen Minh Hoang", "May Tinh", 12, 6.7),
    (2, "Tran Thi Lan", "Kinh Te", 34, 9.2),
    (3, "Pham Van Nam", "Toan Tin", None, 7.9),
    (4, "Le Thanh Huyen", "Toan Tin", 20, 7.2),
    (5, "Vu Quoc Anh", "May Tinh", 24, 8.0),
    (6, "Dang Thuy Linh", "May Tinh", 24, 5.5),
    (7, "Bui Tien Dung", "Kinh Te", 34, 9.2),
    (8, "Ho Ngoc Mai", "Toan Tin", 20, 8.8),
    (9, "Duong Huu Phuc", "Kinh Te", None, 7.2),
    (10, "Cao Thi Hanh", "May Tinh", None, 7.0)
]

# Dữ liệu cho bảng course
course_data = [
    (12, "Giai tich"),
    (34, "Thong ke"),
    (26, "Tin hoc")
]

# Chèn dữ liệu vào bảng student
cursor.executemany("INSERT INTO student VALUES (?, ?, ?, ?, ?);", student_data)

# Chèn dữ liệu vào bảng course
cursor.executemany("INSERT INTO course VALUES (?, ?);", course_data)

conn.commit()


In [4]:
# Đọc dữ liệu từ bảng student
student_df = pd.read_sql_query("SELECT * FROM student;", conn)
print("Bảng Student:")
display(student_df)

# Đọc dữ liệu từ bảng course
course_df = pd.read_sql_query("SELECT * FROM course;", conn)
print("Bảng Course:")
display(course_df)

Bảng Student:


,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7
1,2,Tran Thi Lan,Kinh Te,34.0,9.2
2,3,Pham Van Nam,Toan Tin,NaN,7.9
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2
4,5,Vu Quoc Anh,May Tinh,24.0,8.0
5,6,Dang Thuy Linh,May Tinh,24.0,5.5
6,7,Bui Tien Dung,Kinh Te,34.0,9.2
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2
9,10,Cao Thi Hanh,May Tinh,NaN,7.0


Bảng Course:


,id,course_name
0,12,Giai tich
1,26,Tin hoc
2,34,Thong ke


 Nhận xét

**Đối với bảng Student:**

- Bảng gồm 10 dòng và 5 cột.
- Các cột bao gồm:
  - student_id: mã sinh viên (kiểu số nguyên)
  - name: tên sinh viên (kiểu chuỗi)
  - class: lớp học (kiểu chuỗi)
  - course_id: mã môn học (kiểu số, có giá trị thiếu)
  - score: điểm số (kiểu số thực)
- Cột course_id có 3 giá trị bị thiếu (NaN) tại các sinh viên có student_id là 3, 9 và 10.
- Việc thiếu dữ liệu này có thể gây khó khăn trong việc liên kết với bảng Course và ảnh hưởng đến kết quả phân tích.

**Đối với bảng Course:**

- Bảng gồm 3 dòng và 2 cột.
- Các cột bao gồm:
  - id: mã môn học (kiểu số nguyên)
  - course_name: tên môn học (kiểu chuỗi)
- Dữ liệu trong bảng Course đầy đủ, không có giá trị thiếu.



#1. Kết nối hai bảng


## 1.1 Sử dụng tích Decartes

In [5]:
print("Tích Decartes:")
cursor.execute('''
SELECT * FROM student, course;
''')
df_decartes = pd.DataFrame(cursor.fetchall(), columns=['student_id', 'name', 'class', 'course_id', 'score', 'id', 'course_name'])
print(df_decartes)
print("\n")

Tích Decartes:
    student_id               name     class  course_id  score  id course_name
0            1  Nguyen Minh Hoang  May Tinh       12.0    6.7  12   Giai tich
1            1  Nguyen Minh Hoang  May Tinh       12.0    6.7  26     Tin hoc
2            1  Nguyen Minh Hoang  May Tinh       12.0    6.7  34    Thong ke
3            2       Tran Thi Lan   Kinh Te       34.0    9.2  12   Giai tich
4            2       Tran Thi Lan   Kinh Te       34.0    9.2  26     Tin hoc
5            2       Tran Thi Lan   Kinh Te       34.0    9.2  34    Thong ke
6            3       Pham Van Nam  Toan Tin        NaN    7.9  12   Giai tich
7            3       Pham Van Nam  Toan Tin        NaN    7.9  26     Tin hoc
8            3       Pham Van Nam  Toan Tin        NaN    7.9  34    Thong ke
9            4     Le Thanh Huyen  Toan Tin       20.0    7.2  12   Giai tich
10           4     Le Thanh Huyen  Toan Tin       20.0    7.2  26     Tin hoc
11           4     Le Thanh Huyen  Toan Tin      

 Nhận xét

- Bảng kết quả có tổng cộng 30 dòng, đúng bằng 10 sinh viên × 3 môn học trong bảng course.
- Mỗi sinh viên xuất hiện lặp lại 3 lần, tương ứng với 3 môn học: Giải tích, Tin học và Thống kê.
- Ví dụ: sinh viên Nguyen Minh Hoang (student_id = 1) được ghép với cả 3 môn, mặc dù course_id ban đầu chỉ là 12.
- Các sinh viên có course_id bị thiếu (NaN) như student_id 3, 9, 10 vẫn được ghép đầy đủ với cả 3 môn học.
- Các giá trị course_id bị thiếu (NaN) vẫn xuất hiện đầy đủ trong các tổ hợp, cho thấy phép nối này không phụ thuộc vào điều kiện liên kết.
- Giá trị course_id trong bảng student không khớp với id của bảng course trong nhiều dòng, chứng tỏ mối quan hệ giữa sinh viên và môn học không được phản ánh chính xác.
- Cột score bị lặp lại cho mỗi tổ hợp, mặc dù trên thực tế mỗi sinh viên chỉ có điểm ở môn đã đăng ký.

=> Như vậy, phép tích Descartes không phù hợp để phân tích dữ liệu thực tế vì tạo ra nhiều bản ghi không chính xác và dư thừa.


##1.2: Sử dụng JOIN: INNER JOIN, LEFT JOIN, RIGHT JOIN, FULL OUTER JOIN.

###1.2.1: INNER JOIN

In [6]:
print("INNER JOIN:")
cursor.execute('''
SELECT * FROM student INNER JOIN course ON student.course_id = course.id;
''')
df_inner = pd.DataFrame(cursor.fetchall(), columns=['student_id', 'name', 'class', 'course_id', 'score', 'id', 'course_name'])
print(df_inner)
print("\n")

INNER JOIN:
   student_id               name     class  course_id  score  id course_name
0           1  Nguyen Minh Hoang  May Tinh         12    6.7  12   Giai tich
1           2       Tran Thi Lan   Kinh Te         34    9.2  34    Thong ke
2           7      Bui Tien Dung   Kinh Te         34    9.2  34    Thong ke




Kết luận:
- Bảng chỉ hiển thị các dòng có giá trị khớp giữa course_id của bảng students và id của bảng courses.
- Chỉ có 3 sinh viên xuất hiện, cho thấy rất ít sinh viên có đăng ký môn học trùng khớp giữa hai bảng.

###1.2.2: LEFT JOIN

In [7]:
print("LEFT JOIN:")
cursor.execute('''
SELECT * FROM student LEFT JOIN course ON student.course_id = course.id;
''')
df_left = pd.DataFrame(cursor.fetchall(), columns=['student_id', 'name', 'class', 'course_id', 'score', 'id', 'course_name'])
print(df_left)
print("\n")

LEFT JOIN:
   student_id               name     class  course_id  score    id course_name
0           1  Nguyen Minh Hoang  May Tinh       12.0    6.7  12.0   Giai tich
1           2       Tran Thi Lan   Kinh Te       34.0    9.2  34.0    Thong ke
2           3       Pham Van Nam  Toan Tin        NaN    7.9   NaN        None
3           4     Le Thanh Huyen  Toan Tin       20.0    7.2   NaN        None
4           5        Vu Quoc Anh  May Tinh       24.0    8.0   NaN        None
5           6     Dang Thuy Linh  May Tinh       24.0    5.5   NaN        None
6           7      Bui Tien Dung   Kinh Te       34.0    9.2  34.0    Thong ke
7           8        Ho Ngoc Mai  Toan Tin       20.0    8.8   NaN        None
8           9     Duong Huu Phuc   Kinh Te        NaN    7.2   NaN        None
9          10       Cao Thi Hanh  May Tinh        NaN    7.0   NaN        None




 Kết luận

- LEFT JOIN giữ lại toàn bộ sinh viên và chỉ ghép thêm thông tin môn học khi có giá trị khớp.  
- Các giá trị NaN ở cột id và course_name cho thấy một số sinh viên có course_id không hợp lệ hoặc không tồn tại trong bảng course.

###1.2.3: RIGHT JOIN

In [8]:
print("RIGHT JOIN:")
cursor.execute('''
SELECT * FROM course LEFT JOIN student ON course.id = student.course_id;
''')
df_right = pd.DataFrame(cursor.fetchall(), columns=['id', 'course_name', 'student_id', 'name', 'class', 'course_id', 'score'])
print(df_right)
print("\n")

RIGHT JOIN:
   id course_name  student_id               name     class  course_id  score
0  12   Giai tich         1.0  Nguyen Minh Hoang  May Tinh       12.0    6.7
1  26     Tin hoc         NaN               None      None        NaN    NaN
2  34    Thong ke         7.0      Bui Tien Dung   Kinh Te       34.0    9.2
3  34    Thong ke         2.0       Tran Thi Lan   Kinh Te       34.0    9.2




###1.2.4: FULL OUTER JOIN

In [9]:
query = '''
SELECT * FROM student LEFT JOIN course
ON student.course_id = course.id
UNION ALL
SELECT * FROM course LEFT JOIN student
ON course.id = student.course_id
WHERE student.course_id IS NULL;
'''
df = pd.read_sql_query(query, conn)
print("FULL OUTER JOIN: ", df)

FULL OUTER JOIN:      student_id               name     class  course_id  score    id  \
0            1  Nguyen Minh Hoang  May Tinh       12.0    6.7  12.0   
1            2       Tran Thi Lan   Kinh Te       34.0    9.2  34.0   
2            3       Pham Van Nam  Toan Tin        NaN    7.9   NaN   
3            4     Le Thanh Huyen  Toan Tin       20.0    7.2   NaN   
4            5        Vu Quoc Anh  May Tinh       24.0    8.0   NaN   
5            6     Dang Thuy Linh  May Tinh       24.0    5.5   NaN   
6            7      Bui Tien Dung   Kinh Te       34.0    9.2  34.0   
7            8        Ho Ngoc Mai  Toan Tin       20.0    8.8   NaN   
8            9     Duong Huu Phuc   Kinh Te        NaN    7.2   NaN   
9           10       Cao Thi Hanh  May Tinh        NaN    7.0   NaN   
10          26            Tin hoc      None        NaN    NaN   NaN   

   course_name  
0    Giai tich  
1     Thong ke  
2         None  
3         None  
4         None  
5         None  
6     Thon

Kết luận:
- FULL OUTER JOIN giữ lại tất cả các bản ghi từ cả hai bảng students và courses, điền NaN khi không có giá trị khớp.
- Bảng có dòng với id = 26 và course_name = "Tin hoc" nhưng không có sinh viên nào đăng ký, thể hiện dữ liệu bị thiếu hoặc không có sinh viên nào học môn này.

# 2. Hãy cập nhật những giá trị course_id còn thiếu trong bảng student bằng câu lệnh SQL, trong đó các giá trị được điền là những giá trị nằm trong bảng course và loại bỏ những bản ghi tham gia những môn học không tồn tại bảng course.

In [10]:
# Cập nhật course_id còn thiếu bằng giá trị ngẫu nhiên từ bảng course
cursor.execute('''
UPDATE student
SET course_id = (
    SELECT id FROM course ORDER BY RANDOM() LIMIT 1
)
WHERE course_id IS NULL;
''')
conn.commit()

# Loại bỏ các bản ghi có course_id không hợp lệ
cursor.execute('''
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course);
''')
conn.commit()

# Kiểm tra lại dữ liệu
cursor.execute('SELECT * FROM student;')
print(pd.DataFrame(cursor.fetchall(), columns=['student_id', 'name', 'class', 'course_id', 'score']))


   student_id               name     class  course_id  score
0           1  Nguyen Minh Hoang  May Tinh         12    6.7
1           2       Tran Thi Lan   Kinh Te         34    9.2
2           3       Pham Van Nam  Toan Tin         26    7.9
3           7      Bui Tien Dung   Kinh Te         34    9.2
4           9     Duong Huu Phuc   Kinh Te         26    7.2
5          10       Cao Thi Hanh  May Tinh         26    7.0


Kết luận:
- Các dòng có course_id là NaN trước đó đã được thay thế bằng mã môn học ngẫu nhiên từ bảng course.
- Các dòng có course_id không tồn tại trong bảng course đã bị xóa.
- Bảng kết quả chỉ còn lại các sinh viên có course_id hợp lệ và không còn giá trị NaN.

## 2.1: a. Tổng số sinh viên, điểm trung bình của từng lớp.

In [11]:
# Truy vấn tổng số sinh viên và điểm trung bình của từng lớp
query = '''
SELECT class, COUNT(student_id) AS total_students, ROUND(AVG(score), 2) AS avg_score
FROM student
GROUP BY class;
'''

df = pd.read_sql_query(query, conn)
print(df)

      class  total_students  avg_score
0   Kinh Te               3       8.53
1  May Tinh               2       6.85
2  Toan Tin               1       7.90


 Kết luận

- Lớp Kinh Tế có điểm trung bình cao nhất (8.53), cho thấy chất lượng học tập của lớp này tương đối tốt và ổn định.
- Lớp Máy Tính có điểm trung bình thấp nhất (6.85), cho thấy kết quả học tập còn hạn chế và cần được cải thiện.
- Lớp Toán Tin có điểm trung bình ở mức khá (7.90), tuy nhiên số lượng sinh viên ít nên kết quả này có thể chưa phản ánh chính xác toàn bộ chất lượng của lớp.

=> Nhìn chung, có sự chênh lệch về kết quả học tập giữa các lớp, trong đó lớp Kinh Tế nổi bật hơn, còn lớp Máy Tính cần có biện pháp nâng cao chất lượng học tập.

##2.2: b. Tổng số sinh viên, điểm trung bình của từng môn học.



In [12]:
# Truy vấn tổng số sinh viên và điểm trung bình của từng môn học
query = '''
SELECT course.course_name, COUNT(student.student_id) AS total_students, ROUND(AVG(student.score), 2) AS avg_score
FROM student
LEFT JOIN course ON student.course_id = course.id
GROUP BY course.course_name;
'''

df = pd.read_sql_query(query, conn)
print(df)

  course_name  total_students  avg_score
0   Giai tich               1       6.70
1    Thong ke               2       9.20
2     Tin hoc               3       7.37



Kết luận

- Môn Thống kê có điểm trung bình cao nhất, cho thấy kết quả học tập của sinh viên ở môn này là tốt nhất.
- Môn Giải tích có số lượng sinh viên tham gia ít hơn và điểm trung bình chưa cao, cho thấy cần cải thiện cả về mức độ tham gia và chất lượng học tập.
- Môn Tin học có số lượng sinh viên tham gia nhiều nhưng điểm trung bình chưa cao, cho thấy cần nâng cao hiệu quả học tập của sinh viên ở môn này.


##2.3: c. Phân loại thi đua theo số điểm của từng môn học biết:

- i. Điểm TB ≥  9.0: Xuất sắc.
- ii. 6.0 ≤ Điểm TB ≤ 8.9: Tốt.
- iii. Điểm TB < 6.0: Kém.

In [13]:
# Truy vấn phân loại thi đua theo điểm trung bình của từng môn học
query = '''
SELECT course.course_name,
       ROUND(AVG(student.score), 2) AS avg_score,
       CASE
           WHEN ROUND(AVG(student.score), 2) >= 9.0 THEN 'Xuất sắc'
           WHEN ROUND(AVG(student.score), 2) BETWEEN 6.0 AND 8.9 THEN 'Tốt'
           ELSE 'Kém'
       END AS rating
FROM student
LEFT JOIN course ON student.course_id = course.id
GROUP BY course.course_name;
'''

df = pd.read_sql_query(query, conn)
print(df)

  course_name  avg_score    rating
0   Giai tich       6.70       Tốt
1    Thong ke       9.20  Xuất sắc
2     Tin hoc       7.37       Tốt


Kết luận

- Môn Thống kê đạt xếp loại Xuất sắc với điểm trung bình cao nhất, thể hiện chất lượng học tập nổi bật.
- Hai môn Giải tích và Tin học được xếp loại Tốt, tuy nhiên vẫn cần có các biện pháp cải thiện để nâng cao kết quả học tập.
- Không có môn nào thuộc mức Kém, cho thấy chất lượng đào tạo hiện tại ở mức tương đối tốt.


#3. Hãy xếp hạng sinh viên thông qua:

**a. Điểm số.**

**b. Điểm số theo lớp học.**

**c. Điểm số theo mã môn học.**

**và cho biết top 3 sinh viện đạt thứ hạng cao nhất, top 3 sinh viên đạt thứ hạng thấp nhất theo
từng trường hợp trên.**

##3.1: a. Điểm số.

In [14]:
# Truy vấn xếp hạng sinh viên theo điểm số
query = '''
SELECT student_id, name, class, course_id, score,
       RANK() OVER (ORDER BY score DESC) AS rank
FROM student
ORDER BY score DESC;
'''

df_rank = pd.read_sql_query(query, conn)
print(df_rank)

   student_id               name     class  course_id  score  rank
0           2       Tran Thi Lan   Kinh Te         34    9.2     1
1           7      Bui Tien Dung   Kinh Te         34    9.2     1
2           3       Pham Van Nam  Toan Tin         26    7.9     3
3           9     Duong Huu Phuc   Kinh Te         26    7.2     4
4          10       Cao Thi Hanh  May Tinh         26    7.0     5
5           1  Nguyen Minh Hoang  May Tinh         12    6.7     6


 Kết luận

- Lớp Kinh Tế có thành tích nổi bật khi có sinh viên đạt hạng cao (hạng 1) và duy trì các vị trí tốt khác, cho thấy chất lượng học tập tương đối ổn định.
- Lớp Máy Tính có xu hướng điểm thấp hơn so với các lớp còn lại, phản ánh kết quả học tập chưa cao và cần có biện pháp cải thiện.
- Không có sinh viên nào có điểm dưới 6.5, cho thấy mặt bằng học lực chung của sinh viên ở mức khá trở lên.



## 3.2: b. Điểm số theo lớp học.

In [15]:
# Truy vấn xếp hạng sinh viên theo điểm số trong từng lớp
query = '''
SELECT student_id, name, class, course_id, score,
       RANK() OVER (PARTITION BY class ORDER BY score DESC) AS class_rank
FROM student
ORDER BY class, class_rank;
'''

df_class_rank = pd.read_sql_query(query, conn)
print(df_class_rank)

   student_id               name     class  course_id  score  class_rank
0           2       Tran Thi Lan   Kinh Te         34    9.2           1
1           7      Bui Tien Dung   Kinh Te         34    9.2           1
2           9     Duong Huu Phuc   Kinh Te         26    7.2           3
3          10       Cao Thi Hanh  May Tinh         26    7.0           1
4           1  Nguyen Minh Hoang  May Tinh         12    6.7           2
5           3       Pham Van Nam  Toan Tin         26    7.9           1


 Kết luận

- Lớp Kinh Tế có mức độ cạnh tranh cao khi có sự chênh lệch rõ rệt giữa các thứ hạng, phản ánh sự phân hóa về năng lực học tập trong lớp.
- Lớp Máy Tính có điểm số chưa thực sự nổi bật ở nhóm dẫn đầu, cho thấy cần nâng cao chất lượng học tập chung.
- Lớp Toán Tin có mặt bằng điểm ở mức khá, tuy nhiên số lượng sinh viên ít nên mức độ cạnh tranh chưa cao.


##3.3: c. Điểm số theo mã môn học.

In [16]:
# Truy vấn xếp hạng sinh viên theo điểm số trong từng mã môn học
query = '''
SELECT student_id, name, class, course_id, score,
       RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS course_rank
FROM student
ORDER BY course_id, course_rank;
'''

df_course_rank = pd.read_sql_query(query, conn)
print(df_course_rank)

   student_id               name     class  course_id  score  course_rank
0           1  Nguyen Minh Hoang  May Tinh         12    6.7            1
1           3       Pham Van Nam  Toan Tin         26    7.9            1
2           9     Duong Huu Phuc   Kinh Te         26    7.2            2
3          10       Cao Thi Hanh  May Tinh         26    7.0            3
4           2       Tran Thi Lan   Kinh Te         34    9.2            1
5           7      Bui Tien Dung   Kinh Te         34    9.2            1


##3.4: Cho biết top 3 sinh viện đạt thứ hạng cao nhất, top 3 sinh viên đạt thứ hạng thấp nhất theo từng trường hợp trên.

In [17]:
# Truy vấn top 3 cao nhất và thấp nhất theo điểm số toàn bộ
query = '''
WITH ranked_students AS (
    SELECT student_id, name, class, course_id, score,
           RANK() OVER (ORDER BY score DESC) AS rank_desc,
           RANK() OVER (ORDER BY score ASC) AS rank_asc
    FROM student
)
SELECT * FROM ranked_students
WHERE rank_desc <= 3 OR rank_asc <= 3
ORDER BY rank_desc;
'''
df_top_all = pd.read_sql_query(query, conn)
print(" Top 3 cao nhất & thấp nhất theo điểm số toàn bộ:")
print(df_top_all)
print("\n")


 Top 3 cao nhất & thấp nhất theo điểm số toàn bộ:
   student_id               name     class  course_id  score  rank_desc  \
0           2       Tran Thi Lan   Kinh Te         34    9.2          1   
1           7      Bui Tien Dung   Kinh Te         34    9.2          1   
2           3       Pham Van Nam  Toan Tin         26    7.9          3   
3           9     Duong Huu Phuc   Kinh Te         26    7.2          4   
4          10       Cao Thi Hanh  May Tinh         26    7.0          5   
5           1  Nguyen Minh Hoang  May Tinh         12    6.7          6   

   rank_asc  
0         5  
1         5  
2         4  
3         3  
4         2  
5         1  




Top 3 cao nhất:
- Trần Thị Lan và Bùi Tiến Dũng (9.2 điểm) — Hạng 1.
- Phạm Văn Nam (7.9 điểm) — Hạng 3.

Top 3 thấp nhất:
- Nguyễn Minh Hoàng (6.7 điểm) — Hạng 6 (thấp nhất).
- Cao Thị Hạnh (7.0 điểm) — Hạng 5.
- Dương Hữu Phúc (7.2 điểm) — Hạng 4.

In [18]:
# Truy vấn top 3 cao nhất và thấp nhất theo điểm từng lớp
query = '''
WITH ranked_students AS (
    SELECT student_id, name, class, course_id, score,
           RANK() OVER (PARTITION BY class ORDER BY score DESC) AS class_rank_desc,
           RANK() OVER (PARTITION BY class ORDER BY score ASC) AS class_rank_asc
    FROM student
)
SELECT * FROM ranked_students
WHERE class_rank_desc <= 3 OR class_rank_asc <= 3
ORDER BY class, class_rank_desc;
'''

df_top_class = pd.read_sql_query(query, conn)
print(" Top 3 cao nhất & thấp nhất theo điểm số từng lớp:")
print(df_top_class)
print("\n")


 Top 3 cao nhất & thấp nhất theo điểm số từng lớp:
   student_id               name     class  course_id  score  class_rank_desc  \
0           2       Tran Thi Lan   Kinh Te         34    9.2                1   
1           7      Bui Tien Dung   Kinh Te         34    9.2                1   
2           9     Duong Huu Phuc   Kinh Te         26    7.2                3   
3          10       Cao Thi Hanh  May Tinh         26    7.0                1   
4           1  Nguyen Minh Hoang  May Tinh         12    6.7                2   
5           3       Pham Van Nam  Toan Tin         26    7.9                1   

   class_rank_asc  
0               2  
1               2  
2               1  
3               2  
4               1  
5               1  




 Top 3 cao nhất & thấp nhất theo điểm số từng lớp

Lớp Kinh Tế:
- Cao nhất: Trần Thị Lan và Bùi Tiến Dũng (9.2 điểm) — Đồng hạng 1.
- Thấp nhất: Dương Hữu Phúc (7.2 điểm) — Hạng 3.

Lớp Máy Tính:
- Cao nhất: Cao Thị Hạnh (7.0 điểm) — Hạng 1.
- Thấp nhất: Nguyễn Minh Hoàng (6.7 điểm) — Hạng 2 (cũng là thấp nhất).

Lớp Toán Tin:
- Cao nhất: Phạm Văn Nam (7.9 điểm) — Hạng 1.
- Thấp nhất: Cũng là Phạm Văn Nam (7.9 điểm) — Hạng 1 (vì chỉ có 1 sinh viên).

In [22]:
# Truy vấn top 3 cao nhất và thấp nhất theo điểm từng môn học
query = '''
WITH ranked_students AS (
    SELECT student_id, name, class, course_id, score,
           RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS course_rank_desc,
           RANK() OVER (PARTITION BY course_id ORDER BY score ASC) AS course_rank_asc
    FROM student
)
SELECT * FROM ranked_students
WHERE course_rank_desc <= 3 OR course_rank_asc <= 3
ORDER BY course_id, course_rank_desc;
'''

df_top_course = pd.read_sql_query(query, conn)
print(" Top 3 cao nhất & thấp nhất theo điểm số từng môn học:")
print(df_top_course)
print("\n")


 Top 3 cao nhất & thấp nhất theo điểm số từng môn học:
   student_id               name     class  course_id  score  \
0           1  Nguyen Minh Hoang  May Tinh         12    6.7   
1           3       Pham Van Nam  Toan Tin         26    7.9   
2           9     Duong Huu Phuc   Kinh Te         26    7.2   
3          10       Cao Thi Hanh  May Tinh         26    7.0   
4           2       Tran Thi Lan   Kinh Te         34    9.2   
5           7      Bui Tien Dung   Kinh Te         34    9.2   

   course_rank_desc  course_rank_asc  
0                 1                1  
1                 1                3  
2                 2                2  
3                 3                1  
4                 1                1  
5                 1                1  




##4. Hãy bổ sung thêm một trường graduation_date có kiểu dữ liệu là DATETIME vào bảng student để xác định thời gian tốt nghiệp của từng bạn, trong đó thời gian tốt nghiệp được xác định bởi thời gian hiện tại cộng với số hạng tương ứng của bạn đó tính theo điểm số.

In [23]:
# Thêm cột graduation_date với kiểu DATETIME nếu chưa có
cursor.execute('''
ALTER TABLE student ADD COLUMN graduation_date DATETIME;
''')
conn.commit()

In [24]:
# Cập nhật graduation_date dựa trên thứ hạng
cursor.execute('''
WITH ranked_students AS (
    SELECT student_id, RANK() OVER (ORDER BY score DESC) AS rank
    FROM student
)
UPDATE student
SET graduation_date = DATETIME('now', '+' || (
    SELECT rank FROM ranked_students WHERE ranked_students.student_id = student.student_id
) || ' months');
''')
conn.commit()

In [25]:
# Truy vấn lại để kiểm tra kết quả
query = '''
SELECT student_id, name, class, course_id, score, graduation_date
FROM student
ORDER BY score DESC;
'''

df_updated = pd.read_sql_query(query, conn)
print(" Bảng `student` sau khi thêm trường `graduation_date`:")
print(df_updated)


 Bảng `student` sau khi thêm trường `graduation_date`:
   student_id               name     class  course_id  score  \
0           2       Tran Thi Lan   Kinh Te         34    9.2   
1           7      Bui Tien Dung   Kinh Te         34    9.2   
2           3       Pham Van Nam  Toan Tin         26    7.9   
3           9     Duong Huu Phuc   Kinh Te         26    7.2   
4          10       Cao Thi Hanh  May Tinh         26    7.0   
5           1  Nguyen Minh Hoang  May Tinh         12    6.7   

       graduation_date  
0  2026-05-02 06:03:16  
1  2026-05-02 06:03:16  
2  2026-07-02 06:03:16  
3  2026-08-02 06:03:16  
4  2026-09-02 06:03:16  
5  2026-10-02 06:03:16  


 Nhận xét

- Sau khi thêm trường `graduation_date`, mỗi sinh viên được gán một thời điểm tốt nghiệp dựa trên điểm số.
- Những sinh viên có điểm cao như Trần Thị Lan và Bùi Tiến Dũng (9.2) có ngày tốt nghiệp sớm nhất (2026-05-02).
- Các sinh viên có điểm trung bình như Phạm Văn Nam (7.9), Dương Hữu Phúc (7.2), Cao Thị Hạnh (7.0) có thời gian tốt nghiệp muộn hơn.
- Sinh viên có điểm thấp nhất trong bảng là Nguyễn Minh Hoàng (6.7) có ngày tốt nghiệp muộn nhất (2026-10-02).
- Thời gian tốt nghiệp tăng dần theo chiều giảm của điểm số, cho thấy mối quan hệ ngược giữa điểm và thời gian hoàn thành.

